In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [ ]:
!pip install datasets
from datasets import load_dataset

ds = load_dataset("BoKelvin/SLAKE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.6 MB/s eta 0:00:00


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

validation.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9835 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2099 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2094 [00:00<?, ? examples/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/model_ilab/data/imgs


In [ ]:
%cd /content/drive/MyDrive/model_ilab/data/imgs

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['img_name', 'location', 'answer', 'modality', 'base_type', 'answer_type', 'question', 'qid', 'content_type', 'triple', 'img_id', 'q_lang'],
        num_rows: 9835
    })
    validation: Dataset({
        features: ['img_name', 'location', 'answer', 'modality', 'base_type', 'answer_type', 'question', 'qid', 'content_type', 'triple', 'img_id', 'q_lang'],
        num_rows: 2099
    })
    test: Dataset({
        features: ['img_name', 'location', 'answer', 'modality', 'base_type', 'answer_type', 'question', 'qid', 'content_type', 'triple', 'img_id', 'q_lang'],
        num_rows: 2094
    })
})

In [ ]:
train_df = pd.DataFrame(ds['train'])
validation_df = pd.DataFrame(ds['validation'])
test_df = pd.DataFrame(ds['test'])

In [ ]:
# select only q_lang is 'en'
train_df = train_df[train_df['q_lang'] == 'en']
validation_df = validation_df[validation_df['q_lang'] == 'en']
test_df = test_df[test_df['q_lang'] == 'en']

In [ ]:
df1 = pd.concat([train_df, validation_df, test_df])

In [ ]:
df1['is_radiology'] = 1
df1 = df1[['img_name', 'is_radiology']]

In [ ]:
df1.head()

,img_name,is_radiology
0,xmlab1/source.jpg,1
1,xmlab1/source.jpg,1
2,xmlab1/source.jpg,1
3,xmlab1/source.jpg,1
4,xmlab1/source.jpg,1


In [ ]:
import gdown
import os
import zipfile

In [ ]:
def dataset(directory, id):
  image_url = f'https://drive.google.com/uc?id={id}'
  zip_path = 'dataset.zip'

  # Directory to extract the ZIP contents
  extract_to = f'{directory}/'

  # Download the ZIP file
  gdown.download(image_url, zip_path, quiet=False)

  # Extract ZIP file
  if os.path.exists(zip_path):
      try:
          with zipfile.ZipFile(zip_path, 'r') as zip_ref:
              zip_ref.extractall(extract_to)
          print("Extraction was successful")
          os.remove(zip_path)
      except zipfile.BadZipFile:
          print("Failed to extract: the downloaded file is not a ZIP file or is corrupt")
  else:
      print("File does not exist, check the download URL and process.")

In [ ]:
img_id = '1hG7oCfO7lOajOUmThTHry5COVqwxbKHU' #Google drive id for Image Dataset
text_id = '1WTjo_InLaB23RUzIpPKrsm4ZxrEjbpQZ' #Google drive id for text Dataset

#Downloading dataset
dataset('dataset', img_id)
dataset('text', text_id)

Downloading...
From (original): https://drive.google.com/uc?id=1hG7oCfO7lOajOUmThTHry5COVqwxbKHU
From (redirected): https://drive.google.com/uc?id=1hG7oCfO7lOajOUmThTHry5COVqwxbKHU&confirm=t&uuid=c9ea44bd-4511-4037-8011-28736726b404
To: /content/drive/MyDrive/model_ilab/data/imgs/dataset.zip
100%|██████████| 1.12G/1.12G [00:24<00:00, 46.3MB/s]


Extraction was successful


Downloading...
From: https://drive.google.com/uc?id=1WTjo_InLaB23RUzIpPKrsm4ZxrEjbpQZ
To: /content/drive/MyDrive/model_ilab/data/imgs/dataset.zip
100%|██████████| 2.34M/2.34M [00:00<00:00, 185MB/s]

Extraction was successful


In [ ]:
# Defining path to the dataset and caption
img_path = 'dataset/Flicker8k_Dataset'
caption_path = 'text/Flickr8k.token.txt'

In [ ]:
import pandas as pd

data_chunks = []
chunk_size = 1000
temp_data = []
max_rows = 15000
row_count = 0

# Reading the text file
with open(caption_path, 'r', encoding='utf-8') as file:
    for line in file:
        if row_count >= max_rows:  # Stop when we reach the 15,000th row
            break

        parts = line.split('#')  # Splitting the line at '#'
        first_col = parts[0].strip()  # Image name

        # Handle the middle part (caption), checking if there's a tab
        middle_part = parts[1].split('\t')[1].strip() if '\t' in parts[1] else parts[1].strip()

        # Handle the last part (in case there is another '#' in the caption)
        last_part = parts[2].strip() if len(parts) > 2 else ''

        # Combine middle and last part to form the full caption
        second_col = f"{middle_part} {last_part}".strip()

        # Add the current row to temp_data
        temp_data.append([first_col, second_col])
        row_count += 1  # Increment row count

        # Create a chunk dataframe when chunk size is reached
        if len(temp_data) >= chunk_size:
            df_chunk = pd.DataFrame(temp_data, columns=['Image', 'Caption'])
            data_chunks.append(df_chunk)
            temp_data = []

# If there is leftover data in temp_data (less than chunk_size), create the final chunk
if temp_data:
    df_chunk = pd.DataFrame(temp_data, columns=['Image', 'Caption'])
    data_chunks.append(df_chunk)

# Concatenate all chunks to form the final dataframe
df2 = pd.concat(data_chunks, ignore_index=True)

# Display or return the final dataframe
print(df2.shape)  # Prints the shape of the dataframe to verify the size


(15000, 2)


In [ ]:
df2.head()

,Image,Caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


In [ ]:
# checking filename in df mentioned exist or not
def file_exists(filename):
    return os.path.exists(os.path.join(img_path, filename))

# selecting images that exist in folder
df_filtered = df2[df2['Image'].apply(file_exists)]

In [ ]:
df_filtered.head()

,Image,Caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


In [ ]:
df_filtered.shape

(14995, 2)

In [ ]:
# add directory in image
df_filtered['Image'] = df_filtered['Image'].apply(lambda x: f"{img_path}/{x}")

<ipython-input-26-54c5db873d17>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Image'] = df_filtered['Image'].apply(lambda x: f"{img_path}/{x}")


In [ ]:
df_filtered.head()

,Image,Caption
0,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,A child in a pink dress is climbing up a set o...
1,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,A girl going into a wooden building .
2,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,A little girl climbing into a wooden playhouse .
3,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,A little girl climbing the stairs to her playh...
4,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,A little girl in a pink dress going into a woo...


In [ ]:
# df2 rename Imag. to img_name
df_filtered.rename(columns={'Image': 'img_name'}, inplace=True)

<ipython-input-28-15a2f6885f15>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.rename(columns={'Image': 'img_name'}, inplace=True)


In [ ]:
df_filtered['is_radiology'] = 0
df_filtered = df_filtered[['img_name', 'is_radiology']]

<ipython-input-29-88c32bacea9b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['is_radiology'] = 0


In [ ]:
df_filtered.shape

(14995, 2)

In [ ]:
df_filtered.head()

,img_name,is_radiology
0,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,0
1,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,0
2,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,0
3,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,0
4,dataset/Flicker8k_Dataset/1000268201_693b08cb0...,0


In [ ]:
df = pd.concat([df1, df_filtered], ignore_index=True)

In [ ]:
df.head()

,img_name,is_radiology
0,xmlab1/source.jpg,1
1,xmlab1/source.jpg,1
2,xmlab1/source.jpg,1
3,xmlab1/source.jpg,1
4,xmlab1/source.jpg,1


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
class RadiologyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx, 0]  # Image file path
        label = self.dataframe.iloc[idx, 1]     # 0 or 1 (is_radiology)

        # Load image
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


In [ ]:
# Define image transformations (data augmentation and normalization)
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomResizedCrop(299, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Assuming 'df' contains 'img_name' (paths) and 'is_radiology' (0/1 labels)
# Split the data into training and testing
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['is_radiology'], random_state=42)

# Create datasets
train_dataset = RadiologyDataset(train_df, transform=transform)
test_dataset = RadiologyDataset(test_df, transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Use a pre-trained ResNet18 model and modify the final layer for binary classification
model = models.resnet18(pretrained=True)

# Replace the last fully connected layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 1)  # Binary classification (1 output with sigmoid)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 220MB/s]


In [ ]:
# Define the loss function and optimizer
criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy with logits
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from tqdm import tqdm

In [ ]:
def train(model, train_loader, criterion, optimizer, device, epochs=5):
    model.train()  # Set the model to training mode
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"EPOCH : [{epoch+1}/{epochs}]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()  # Zero the parameter gradients

            # Forward pass
            outputs = model(images)
            outputs = outputs.squeeze(1)  # Squeeze the output for compatibility with labels
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Update running loss and accuracy
            running_loss += loss.item() * images.size(0)
            preds = torch.round(torch.sigmoid(outputs))  # Convert logits to binary predictions
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

# Train the model
train(model, train_loader, criterion, optimizer, device, epochs=5)

EPOCH : [1/5]: 100%|██████████| 551/551 [13:00<00:00,  1.42s/it]


Epoch [1/5], Loss: 0.0237, Accuracy: 0.9923


EPOCH : [2/5]: 100%|██████████| 551/551 [05:43<00:00,  1.60it/s]


Epoch [2/5], Loss: 0.0086, Accuracy: 0.9968


EPOCH : [3/5]: 100%|██████████| 551/551 [05:41<00:00,  1.61it/s]


Epoch [3/5], Loss: 0.0152, Accuracy: 0.9947


EPOCH : [4/5]: 100%|██████████| 551/551 [05:37<00:00,  1.63it/s]


Epoch [4/5], Loss: 0.0037, Accuracy: 0.9987


EPOCH : [5/5]: 100%|██████████| 551/551 [05:38<00:00,  1.63it/s]

Epoch [5/5], Loss: 0.0055, Accuracy: 0.9982


In [ ]:
def evaluate(model, test_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient calculation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            outputs = outputs.squeeze(1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.round(torch.sigmoid(outputs))  # Convert logits to binary predictions
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_loss = running_loss / len(test_loader.dataset)
    test_acc = correct / total

    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

# Evaluate the model
evaluate(model, test_loader, criterion, device)

Test Loss: 0.0075, Test Accuracy: 0.9964


In [ ]:
%mkdir /content/drive/MyDrive/models/
%cd /content/drive/MyDrive/models/

/content/drive/MyDrive/models


In [ ]:
# Save the model
torch.save(model.state_dict(), 'radiology_classifier.pth')